# E-Commerce Sales, Customer Behaviour and Business Performance Analysis
## Python EDA (Exploratory Data Analysis)

**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), 9 related CSV files.

**How to use this notebook:** run it cell by cell (Shift + Enter). Read the Markdown before each code cell. After each section, write one or two lines in `documentation/decisions.md` about what you found and decided.

**What this notebook does NOT do:** it does not calculate profit (the dataset has no cost data) and it does not use machine learning (not needed for these business questions).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

# Folder that holds the 9 original CSV files (relative to this notebook)
DATA_DIR = Path("../dataset/original")

## 2. Load the data
`read_csv` loads a file into a DataFrame (a table). We load all nine files and give them short names.

In [ ]:
customers  = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geo        = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
items      = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments   = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
orders     = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products   = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers    = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_t = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

tables = {"customers": customers, "geolocation": geo, "order_items": items,
          "order_payments": payments, "order_reviews": reviews, "orders": orders,
          "products": products, "sellers": sellers, "category_translation": category_t}

# Rows and columns of every table (compare with the counts you wrote in Phase 1)
pd.DataFrame({name: df.shape for name, df in tables.items()}, index=["rows", "columns"]).T

## 3. Understand each table
Look at the first rows, the data types and the basic statistics. Ask for each table: *what does one row represent?*

In [ ]:
orders.head()

In [ ]:
orders.info()

In [ ]:
items.describe()

## 4. Key uniqueness checks
These checks decide which joins are safe. If distinct IDs < rows, the ID repeats, and joining on it can multiply rows.

In [ ]:
checks = pd.DataFrame({
    "table": ["orders", "customers (customer_id)", "customers (customer_unique_id)",
              "order_reviews (order_id)", "order_payments (order_id)"],
    "rows": [len(orders), len(customers), len(customers),
             len(reviews), len(payments)],
    "distinct_ids": [orders["order_id"].nunique(), customers["customer_id"].nunique(),
                     customers["customer_unique_id"].nunique(),
                     reviews["order_id"].nunique(), payments["order_id"].nunique()],
})
checks["repeated_rows"] = checks["rows"] - checks["distinct_ids"]
checks

## 5. Missing values
Missing values are not always mistakes. Missing delivery dates are expected for orders that were never delivered. We check whether the missing values *make sense* before touching them.

In [ ]:
def missing_report(df):
    out = df.isna().sum().to_frame("missing")
    out["pct"] = 100 * out["missing"] / len(df)
    return out[out["missing"] > 0].sort_values("missing", ascending=False)

for name in ["orders", "products", "order_reviews"]:
    print(f"--- {name} ---")
    display(missing_report(tables[name]))

In [ ]:
# Do the missing delivery dates line up with the order status?
orders.groupby("order_status")["order_delivered_customer_date"].apply(lambda s: s.isna().sum()).to_frame("missing_delivery_date") \
      .join(orders["order_status"].value_counts().rename("orders"))

**Write down:** which statuses have missing delivery dates, and does that match what you expect?
(Do not fill these gaps. A missing delivery date on an undelivered order is information, not an error.)

## 6. Duplicates

In [ ]:
for name, df in tables.items():
    print(f"{name:22s} exact duplicate rows: {df.duplicated().sum():,}")

The geolocation table is expected to contain many duplicate rows. We do not use it for the main analysis, so we only note it in the decision log.

## 7. Fix data types and build the clean tables
Dates are loaded as text. We convert them so we can group by month and calculate delivery times.

In [ ]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders["purchase_year"]  = orders["order_purchase_timestamp"].dt.year

orders[date_cols].dtypes

### Business rule: which orders count as sales?
Orders with status `canceled` or `unavailable` did not produce sales, so we exclude them. **Write this decision in your log.**

In [ ]:
orders["order_status"].value_counts()

In [ ]:
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])]

# One row per ORDER ITEM, with order dates, customer and English category name
sales = (items
         .merge(valid_orders[["order_id", "customer_id", "order_status",
                              "order_purchase_timestamp", "purchase_month", "purchase_year"]],
                on="order_id", how="inner")
         .merge(customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
                on="customer_id", how="left")
         .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
         .merge(category_t, on="product_category_name", how="left"))

sales["category"] = sales["product_category_name_english"].fillna(
    sales["product_category_name"]).fillna("unknown")

print("Rows in items:          ", len(items))
print("Rows in sales after joins:", len(sales))

**Join check (very important):** after joining, the number of rows should not be higher than the number of item rows. If it is higher, a join multiplied rows and your revenue would be inflated. Rows can be *lower* because we removed canceled/unavailable orders.

In [ ]:
assert len(sales) <= len(items), "A join multiplied rows! Investigate before continuing."
print("Join check passed.")

## 8. Headline numbers

In [ ]:
total_revenue   = sales["price"].sum()
total_orders    = sales["order_id"].nunique()
total_customers = sales["customer_unique_id"].nunique()
total_products  = sales["product_id"].nunique()
aov             = sales.groupby("order_id")["price"].sum().mean()

kpis = pd.Series({
    "Total revenue (price only)": total_revenue,
    "Total valid orders": total_orders,
    "Total customers (unique people)": total_customers,
    "Products sold": total_products,
    "Average order value": aov,
    "Units sold": len(sales),
})
kpis

**Cross-check:** compare these numbers with the SQL results (`sql/01_basic_analysis.sql`). If SQL and Python disagree, find out why before you trust either. Write what you find in `decisions.md`.

## 9. Sales over time

In [ ]:
monthly = (sales.groupby("purchase_month")
                .agg(revenue=("price", "sum"), orders=("order_id", "nunique"))
                .reset_index())
monthly["mom_growth_pct"] = monthly["revenue"].pct_change() * 100
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["purchase_month"], monthly["revenue"], marker="o")
ax.set_title("Monthly revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

**Careful:** look at the first and last months. They have very few orders, which can create fake growth or fake drops. Say so in your findings instead of quoting those months.

## 10. Product and category performance

In [ ]:
cat_rev = (sales.groupby("category")
                .agg(revenue=("price", "sum"), units=("order_id", "size"))
                .sort_values("revenue", ascending=False))
cat_rev["pct_of_revenue"] = 100 * cat_rev["revenue"] / cat_rev["revenue"].sum()
cat_rev.head(10)

In [ ]:
top10 = cat_rev.head(10).reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top10, y="category", x="revenue", ax=ax, color="#2a6f97")
ax.set_title("Top 10 categories by revenue")
ax.set_xlabel("Revenue")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Categories with meaningful volume but the lowest revenue
cat_rev[cat_rev["units"] >= 50].sort_values("revenue").head(10)

## 11. Customer behaviour

In [ ]:
cust = (sales.groupby("customer_unique_id")
             .agg(total_orders=("order_id", "nunique"), total_spent=("price", "sum"))
             .reset_index())

cust["customer_type"] = np.where(cust["total_orders"] == 1, "One-time", "Repeat")
summary = cust.groupby("customer_type").agg(customers=("customer_unique_id", "count"),
                                            revenue=("total_spent", "sum"))
summary["pct_of_customers"] = 100 * summary["customers"] / summary["customers"].sum()
summary["pct_of_revenue"]   = 100 * summary["revenue"]   / summary["revenue"].sum()
summary

In [ ]:
cust["total_orders"].value_counts().sort_index()

In [ ]:
# How concentrated is spending? Share of revenue by customer quintile
cust_sorted = cust.sort_values("total_spent", ascending=False).reset_index(drop=True)
cust_sorted["quintile"] = pd.qcut(cust_sorted.index, 5, labels=[1, 2, 3, 4, 5])
pareto = cust_sorted.groupby("quintile", observed=True)["total_spent"].sum()
(100 * pareto / pareto.sum()).round(2).to_frame("pct_of_revenue")

**Question for you:** is the "80/20 rule" true here? Do not assume. Read the table above and answer using your own numbers.

## 12. Regional analysis

In [ ]:
state_rev = (sales.groupby("customer_state")
                  .agg(revenue=("price", "sum"), orders=("order_id", "nunique"))
                  .sort_values("revenue", ascending=False))
state_rev["pct_of_revenue"] = 100 * state_rev["revenue"] / state_rev["revenue"].sum()
state_rev.head(10)

## 13. Delivery performance and satisfaction

In [ ]:
delivered = orders[(orders["order_status"] == "delivered") &
                   orders["order_delivered_customer_date"].notna()].copy()
delivered["delivery_days"] = (delivered["order_delivered_customer_date"]
                              - delivered["order_purchase_timestamp"]).dt.days
delivered["is_late"] = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]

print("Average delivery days:", round(delivered["delivery_days"].mean(), 1))
print("Share of late orders (%):", round(100 * delivered["is_late"].mean(), 2))

In [ ]:
# Average review score per ORDER first (an order can have more than one review row)
order_score = reviews.groupby("order_id", as_index=False)["review_score"].mean()
d = delivered.merge(order_score, on="order_id", how="inner")
d.groupby("is_late")["review_score"].agg(["count", "mean"])

This shows an **association** between late delivery and review score. It does not prove cause. Say it that way in interviews.

## 14. Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=items["price"], ax=axes[0], color="#a9d6e5")
axes[0].set_title("Item price")
sns.boxplot(x=delivered["delivery_days"], ax=axes[1], color="#a9d6e5")
axes[1].set_title("Delivery days")
plt.tight_layout()
plt.show()

items["price"].describe()

Expensive items and long delivery times exist in real data. Do not delete outliers automatically. Decide whether they are errors or real, and log your decision.

## 15. Save a clean file for Power BI (optional)
This writes the sales table to `dataset/cleaned/`. It is not required if you connect Power BI to MySQL.

In [ ]:
out_path = Path("../dataset/cleaned")
out_path.mkdir(parents=True, exist_ok=True)

sales["order_date"]  = sales["order_purchase_timestamp"].dt.date
sales["order_month"] = sales["purchase_month"]
sales["order_year"]  = sales["purchase_year"]

sales_out = sales[["order_id", "order_item_id", "order_status", "order_purchase_timestamp",
                   "order_date", "order_month", "order_year",
                   "customer_id", "customer_unique_id", "customer_state", "customer_city",
                   "product_id", "category", "seller_id", "price", "freight_value"]]
sales_out.to_csv(out_path / "sales_clean.csv", index=False)
print("Saved", len(sales_out), "rows")

## 16. Summary template (fill this in with YOUR results)
- Total revenue: ...
- Valid orders: ...
- Real customers: ...
- Share of one-time customers: ...
- Top 3 categories and their share of revenue: ...
- Top 3 states and their share of revenue: ...
- Share of late deliveries and average review score (late vs on time): ...
- Months I excluded from growth statements, and why: ...